# Tensorflow Object Detection API and AWS Sagemaker

In this notebook, you will train and evaluate different models using the [Tensorflow Object Detection API](https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/) and [AWS Sagemaker](https://aws.amazon.com/sagemaker/). 

If you ever feel stuck, you can refer to this [tutorial](https://aws.amazon.com/blogs/machine-learning/training-and-deploying-models-using-tensorflow-2-with-the-object-detection-api-on-amazon-sagemaker/).

## Dataset

We are using the [Waymo Open Dataset](https://waymo.com/open/) for this project. The dataset has already been exported using the tfrecords format. The files have been created following the format described [here](https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/training.html#create-tensorflow-records). You can find data stored on [AWS S3](https://aws.amazon.com/s3/), AWS Object Storage. The images are saved with a resolution of 640x640.

In [1]:
%pip install tensorflow_io "sagemaker>=3,<4" -U

Note: you may need to restart the kernel to use updated packages.


**Important:** after the install cell above finishes, restart the kernel (**Kernel > Restart**) and then run the notebook from the top again. The SageMaker SDK version that was already loaded keeps being used until you restart.

This project uses the SageMaker Python SDK **v3** (`ModelTrainer` for training). SDK v2 classes such as `Estimator`, `TensorFlowModel` and `.deploy()` no longer exist in v3, and this notebook doesn't use them. The next cell checks that v3 is loaded.

In [2]:
from importlib.metadata import version
import sagemaker

sagemaker_version = version("sagemaker")
print(f"SageMaker SDK version: {sagemaker_version}")
# SDK v2 defines sagemaker.__version__, v3 doesn't. If it is still there, v2 is loaded in this kernel.
if not sagemaker_version.startswith("3.") or hasattr(sagemaker, "__version__"):
    raise RuntimeError(
        f"SageMaker SDK {sagemaker_version} is installed, but this notebook needs v3 loaded in the kernel. "
        "Run the install cell above, restart the kernel (Kernel > Restart) and run the notebook from the top again."
    )

SageMaker SDK version: 3.28.0


In [3]:
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData, SourceCode, TensorBoardOutputConfig

Save the IAM role in a variable called `role`. This would be useful when training the model.

In [4]:
role = get_execution_role()
print(role)

arn:aws:iam::299830940405:role/service-role/AmazonSageMaker-ExecutionRole-20261029T130889


In [5]:
import boto3
from botocore.exceptions import ClientError

# The train and val paths below are public S3 buckets created by Udacity for this project
inputs = {'train': 's3://cd2688-object-detection-tf2/train/', 
          'val': 's3://cd2688-object-detection-tf2/val/'} 

# S3 folder for the tensorboard logs. This must be a bucket in your own AWS account.
# By default we use the SageMaker default bucket of your account (it is created automatically if needed).
# To use the bucket you created during setup instead, replace this with 's3://<your-bucket-name>/logs/'
tensorboard_s3_prefix = f's3://{Session().default_bucket()}/logs/'

# Check that the bucket exists and that you can access it before launching the training job
tensorboard_bucket = tensorboard_s3_prefix.split('/')[2]
try:
    boto3.client('s3').head_bucket(Bucket=tensorboard_bucket)
except ClientError as e:
    raise RuntimeError(
        f"Cannot access the S3 bucket '{tensorboard_bucket}' ({e.response['Error']['Code']}). "
        "Use a bucket in your own AWS account for tensorboard_s3_prefix."
    ) from e
print(f"Tensorboard logs will be saved to {tensorboard_s3_prefix}")

Tensorboard logs will be saved to s3://sagemaker-us-east-1-299830940405/logs/


## Container

To train the model, you will first need to build a [docker](https://www.docker.com/) container with all the dependencies required by the TF Object Detection API. The code below does the following:
* clone the Tensorflow models repository
* get the exporter and training scripts from the repository
* build the docker image and push it 
* print the container name

In [6]:
%%bash

# clone the repo and get the scripts
git clone https://github.com/tensorflow/models.git docker/models

# get model_main and exporter_main files from TF2 Object Detection GitHub repository
cp docker/models/research/object_detection/exporter_main_v2.py source_dir 
cp docker/models/research/object_detection/model_main_tf2.py source_dir

fatal: destination path 'docker/models' already exists and is not an empty directory.


In [7]:
# build and push the docker image. This code can be commented out after being run once.
# This will take around 10 mins.
image_name = 'tf2-object-detection'
#!sh ./docker/build_and_push.sh $image_name
!bash -x ./docker/build_and_push.sh $image_name


+ set -euo pipefail
+ image=tf2-object-detection
+++ dirname ./docker/build_and_push.sh
++ cd ./docker
++ pwd
+ SCRIPT_DIR=/home/ec2-user/SageMaker/cd2688-object-detection-in-urban-environment-project/1_model_training/docker
+ '[' '!' -d /home/ec2-user/SageMaker/cd2688-object-detection-in-urban-environment-project/1_model_training/docker/models/research/object_detection ']'
++ aws sts get-caller-identity --query Account --output text
+ ACCOUNT_ID=299830940405
++ aws configure get region
+ AWS_REGION=us-east-1
+ '[' -z us-east-1 ']'
++ date +%Y%m%d%H%M%S
+ TAG=20261103065717
+ fullname=299830940405.dkr.ecr.us-east-1.amazonaws.com/tf2-object-detection:20261103065717
+ aws ecr describe-repositories --repository-names tf2-object-detection
+ echo 'Logging docker in to ECR'
Logging docker in to ECR
+ aws ecr get-login-password --region us-east-1
+ docker login --username AWS --password-stdin 299830940405.dkr.ecr.us-east-1.amazonaws.com
WARNING! Your password will be stored unencrypted in /ho

To verify that the image was correctly pushed to the [Elastic Container Registry](https://aws.amazon.com/ecr/), you can look at it in the AWS webapp. For example, below you can see that three different images have been pushed to ECR. You should only see one, called `tf2-object-detection`.
![ECR Example](../data/example_ecr.png)


In [8]:
# display the container name
with open (os.path.join('docker', 'ecr_image_fullname.txt'), 'r') as f:
    container = f.readlines()[0][:-1]

print(container)

299830940405.dkr.ecr.us-east-1.amazonaws.com/tf2-object-detection:20261103065717


## Pre-trained model from model zoo

As often, we are not training from scratch and we will be using a pretrained model from the TF Object Detection model zoo. You can find pretrained checkpoints [here](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md). Because your time is limited for this project, we recommend to only experiment with the following models:
* SSD MobileNet V2 FPNLite 640x640	
* SSD ResNet50 V1 FPN 640x640 (RetinaNet50)	
* Faster R-CNN ResNet50 V1 640x640	
* EfficientDet D1 640x640	
* Faster R-CNN ResNet152 V1 640x640	

In the code below, the EfficientDet D1 model is downloaded and extracted. This code should be adjusted if you were to experiment with other architectures.

In [9]:
%%bash
mkdir /tmp/checkpoint
mkdir source_dir/checkpoint
wget -O /tmp/efficientdet.tar.gz http://download.tensorflow.org/models/object_detection/tf2/20200711/efficientdet_d1_coco17_tpu-32.tar.gz
tar -zxvf /tmp/efficientdet.tar.gz --strip-components 2 --directory source_dir/checkpoint efficientdet_d1_coco17_tpu-32/checkpoint

mkdir: cannot create directory ‘/tmp/checkpoint’: File exists
mkdir: cannot create directory ‘source_dir/checkpoint’: File exists
--2026-11-03 07:04:55--  http://download.tensorflow.org/models/object_detection/tf2/20200711/efficientdet_d1_coco17_tpu-32.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 172.253.115.207, 172.253.122.207, 172.253.63.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|172.253.115.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51839363 (49M) [application/x-tar]
Saving to: ‘/tmp/efficientdet.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 12.2M 4s
    50K .......... .......... .......... .......... ..........  0% 24.9M 3s
   100K .......... .......... .......... .......... ..........  0% 37.5M 2s
   150K .......... .......... .......... .......... ..........  0% 37.1M 2s
   200K .......... .......... .......... .......... ..........  0%  346M 2s
   250K ........

efficientdet_d1_coco17_tpu-32/checkpoint/ckpt-0.data-00000-of-00001
efficientdet_d1_coco17_tpu-32/checkpoint/checkpoint
efficientdet_d1_coco17_tpu-32/checkpoint/ckpt-0.index


## Edit pipeline.config file

The [`pipeline.config`](source_dir/pipeline.config) in the `source_dir` folder should be updated when you experiment with different models. The different config files are available [here](https://github.com/tensorflow/models/tree/master/research/object_detection/configs/tf2).

>Note: The provided `pipeline.config` file works well with the `EfficientDet` model. You would need to modify it when working with other models.

## Launch Training Job

Now that we have a dataset, a docker image and some pretrained model weights, we can launch the training job. To do so, we create a [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/) (SageMaker Python SDK v3), where we indicate the container name, the training script, the number of training steps etc.

`ModelTrainer` uploads the `source_dir` folder to the training job and runs `run_training.sh` from inside it. The hyperparameters are available in the script as `SM_HP_<NAME>` environment variables. The `run_training.sh` script does the following:
* train the model for `num_train_steps` 
* evaluate over the val dataset
* export the model

Different metrics will be displayed during the evaluation phase, including the mean average precision. These metrics can be used to quantify your model performances and compare over the different iterations.

You can also monitor the training progress by navigating to **Training -> Training Jobs** from the Amazon Sagemaker dashboard in the Web UI.

In [11]:
model_trainer = ModelTrainer(
    role=role,
    training_image=container,
    source_code=SourceCode(
        source_dir='source_dir/',
        entry_script='run_training.sh'
    ),
    hyperparameters={
        "model_dir": "/opt/training",        
        "pipeline_config_path": "pipeline.config",
        "num_train_steps": "2000",    
        "sample_1_of_n_eval_examples": "1"
    },
    compute=Compute(
        instance_count=1,
        instance_type='ml.g5.xlarge'
    ),
    base_job_name='tf2-object-detection'
).with_tensorboard_output_config(
    TensorBoardOutputConfig(
        s3_output_path=tensorboard_s3_prefix,
        local_path='/opt/training/'
    )
)

model_trainer.train(
    input_data_config=[InputData(channel_name=name, data_source=uri) for name, uri in inputs.items()]
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

You should be able to see your model training in the AWS webapp as shown below:
![ECR Example](../data/example_trainings.png)


## Improve on the initial model

Most likely, this initial experiment did not yield optimal results. However, you can make multiple changes to the `pipeline.config` file to improve this model. One obvious change consists in improving the data augmentation strategy. The [`preprocessor.proto`](https://github.com/tensorflow/models/blob/master/research/object_detection/protos/preprocessor.proto) file contains the different data augmentation method available in the Tf Object Detection API. Justify your choices of augmentations in the write-up.

Keep in mind that the following are also available:
* experiment with the optimizer: type of optimizer, learning rate, scheduler etc
* experiment with the architecture. The Tf Object Detection API model zoo offers many architectures. Keep in mind that the pipeline.config file is unique for each architecture and you will have to edit it.
* visualize results on the test frames using the `2_deploy_model` notebook available in this repository.

In the cell below, write down all the different approaches you have experimented with, why you have chosen them and what you would have done if you had more time and resources. Justify your choices using the tensorboard visualizations (take screenshots and insert them in your write-up), the metrics on the evaluation set and the generated animation you have created with [this tool](../2_run_inference/2_deploy_model.ipynb).

In [ ]:
# your write-up goes here.